# 03 - Degrade and augment (CCTV-style, GPU)

Turns the **undegraded** images in `CLEAN_DIR` into the **degraded, class-balanced** training set in
`data/synthetic_degraded/{class}/`, simulating realistic restaurant-camera footage. **Novelty #1.**

**How to use this notebook**
1. (optional) Edit knobs in the **`DEGRADE` settings** cell.
2. Run the **Preview** cell - it degrades 10 fixed images/class (no files written) so you can *instantly* see
   the effect of your changes. Tweak + re-run until it looks right.
3. Run **Generate** to build the full degraded set, then **Balance**, then the final **Visual check**.

**Key properties**
- **GPU, batched, pure PyTorch** (no new packages); falls back to CPU automatically.
- **Class-agnostic / de-biased:** the degradation never sees the label, so no effect can leak the class.
- **Auto-balanced:** smaller classes get *more* augmentations per image, so every class ends up with a
  similar number of images (see the **Generate** cell).
- **Reproducible** (one fixed seed). JPEG applied at save. No "REC"/timestamp HUD is ever drawn.

## Configuration + paths

In [ ]:
import os, math, random, shutil
# --- choose the GPU here (no terminal needed; just press Run) ---
# Set BEFORE importing torch. Change "0" if GPU 0 is busy, then restart the kernel.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

import sys
from pathlib import Path
# Make the local shlomi/utils.py importable whether launched from shlomi/ or the repo root.
for _cand in (Path.cwd(), Path.cwd() / "shlomi"):
    if (_cand / "utils.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand)); break
import utils

torch.set_grad_enabled(False)                       # inference-only preprocessing
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_DIR  = utils.CLEAN_DIR          # undegraded images, per-class subfolders (e.g. Diff_DataSet)
OUTPUT_DIR = utils.DEGRADED_DIR       # data/synthetic_degraded/{class}/
BATCH = 64                            # images per GPU batch (raise if the GPU has spare memory)
SIZE = 224                            # working resolution
AUGS_FOR_LARGEST = 3                  # the BIGGEST class gets this many augs/image; smaller classes get
                                      # proportionally more so the final counts are ~equal (see Generate).

CLASSES = sorted([d.name for d in INPUT_DIR.iterdir() if d.is_dir()]) if INPUT_DIR.is_dir() else []
print("device :", DEVICE)
print("input  :", INPUT_DIR)
print("output :", OUTPUT_DIR)
print("classes:", CLASSES)

def reset_degraded_folder():
    if OUTPUT_DIR.exists():
        print("Deleting existing degraded folder..."); shutil.rmtree(OUTPUT_DIR)
    for cls in CLASSES:
        os.makedirs(OUTPUT_DIR / cls, exist_ok=True)
    print("Fresh degraded folder ready:", OUTPUT_DIR)

## Degradation settings - TUNE HERE

Every knob is in `DEGRADE`. Each effect has a **`prob`** (chance it's applied per image, `0`-`1`) and a
**strength range** (a `(low, high)` pair sampled per image). Each line below also notes the **sensible
range** for that parameter.

**Rule of thumb:** widen / raise a range = harsher; narrow / lower it = gentler; `prob: 0` turns it off.
After editing, run the **Preview** cell to see the effect immediately.

In [ ]:
DEGRADE = {
    # LOW RESOLUTION - shrink to one of these px sizes then scale back to 224 (one size per batch).
    #   values: pixels, each <= 224 (224 = no downscale). Lower = blurrier/pixelated.
    #   sensible: floor ~96 (harsh) ... 160 (mild). The main "it's CCTV" cue. Always applied.
    "downscale_sizes": [128, 160, 192, 224],

    # BLUR (defocus). sigma = blur radius in px.
    #   prob: 0-1   sigma: 0.2 .. 3.0  (0.3-1.5 = decent camera; >2 = noticeably soft)
    "blur":       {"prob": 0.5, "sigma": (0.3, 1.5)},

    # CAMERA TILT (mounted-camera look): per-image rotation + small sideways shift.
    #   max_deg: 0 .. 25  (degrees)        shift: 0.0 .. 0.12  (fraction of frame width)
    "tilt":       {"max_deg": 12, "shift": 0.05},

    # SENSOR NOISE (additive). std on a 0-255 scale; higher = grainier.
    #   prob: 0-1   std: 0 .. 25  (2-10 = decent camera; >15 = low-light/noisy)
    "noise":      {"prob": 0.5, "std": (2, 9)},

    # BRIGHTNESS multiplier. 1.0 = unchanged, <1 darker, >1 brighter.
    #   prob: 0-1   range: 0.5 .. 1.5  (0.8-1.2 = subtle).   <<< too bright? lower the high value >>>
    "brightness": {"prob": 0.5, "range": (0.8, 1.2)},

    # CONTRAST multiplier. 1.0 = unchanged, <1 flatter/greyer, >1 punchier.
    #   prob: 0-1   range: 0.5 .. 1.5  (0.8-1.2 = subtle)
    "contrast":   {"prob": 0.5, "range": (0.8, 1.2)},

    # COLOUR CAST (per-channel gain = white-balance tint). 1.0 = neutral.
    #   prob: 0-1   range: 0.85 .. 1.15  (closer to 1.0 = subtler tint)
    "color_cast": {"prob": 0.4, "range": (0.92, 1.08)},

    # VIGNETTE (dark corners). 0 = none.
    #   prob: 0-1   strength: 0.0 .. 0.6  (0.1-0.3 = mild)
    "vignette":   {"prob": 0.3, "strength": (0.1, 0.3)},

    # JPEG compression (applied at save). quality 1-100; LOWER = more blocky artifacts.
    #   prob: 0-1   quality: 20 .. 95  (45-85 = decent DVR; <35 = heavy blocking)
    "jpeg":       {"prob": 0.6, "quality": (45, 85)},
}

## Degradation functions (GPU, batched)

One small function per effect - each reads its knobs from `DEGRADE`. `degrade_batch` calls them in order
(capture -> lens -> sensor -> output). You normally won't touch these; tune `DEGRADE` instead.

In [ ]:
# --- small helpers ---
def _mask(B, prob, dev):
    """Per-sample on/off switch, shape [B,1,1,1]: 1 with probability `prob`, else 0."""
    return (torch.rand(B, 1, 1, 1, device=dev) < prob).float()

def _gauss_kernel(sigma, device, k=5):
    coords = torch.arange(k, device=device, dtype=torch.float32) - (k - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma * sigma)); g = g / g.sum()
    return (g[:, None] * g[None, :]).expand(3, 1, k, k).contiguous()

def _radial_r2(h, w, device):
    yy = torch.linspace(-1, 1, h, device=device)[:, None]
    xx = torch.linspace(-1, 1, w, device=device)[None, :]
    return ((xx ** 2 + yy ** 2) / 2.0).clamp(0, 1)        # normalized r^2 in [0,1]


# --- one function per effect (all read DEGRADE) ---
def low_resolution(x):
    """Shrink then upscale back -> the core low-res CCTV look (one size per batch)."""
    s = int(random.choice(DEGRADE["downscale_sizes"]))
    H, W = x.shape[-2:]
    x = F.interpolate(x, size=(s, s), mode="bilinear", align_corners=False)
    return F.interpolate(x, size=(H, W), mode="bilinear", align_corners=False)

def blur(x):
    """Gaussian defocus blur (one random sigma per batch)."""
    if random.random() >= DEGRADE["blur"]["prob"]:
        return x
    kern = _gauss_kernel(random.uniform(*DEGRADE["blur"]["sigma"]), x.device)
    return F.conv2d(F.pad(x, (2, 2, 2, 2), mode="reflect"), kern, groups=3)

def camera_tilt(x):
    """Per-sample rotation + sideways shift; reflection border (no black corners)."""
    B = x.shape[0]; dev = x.device
    deg, shift = DEGRADE["tilt"]["max_deg"], DEGRADE["tilt"]["shift"]
    ang = (torch.rand(B, device=dev) * 2 - 1) * (deg * math.pi / 180)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 1] = cos, -sin
    theta[:, 1, 0], theta[:, 1, 1] = sin, cos
    theta[:, 0, 2] = (torch.rand(B, device=dev) * 2 - 1) * shift
    theta[:, 1, 2] = (torch.rand(B, device=dev) * 2 - 1) * shift
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)

def sensor_noise(x):
    """Per-sample additive Gaussian noise (std on a 0-255 scale)."""
    B = x.shape[0]; dev = x.device
    lo, hi = DEGRADE["noise"]["std"]
    std = torch.empty(B, 1, 1, 1, device=dev).uniform_(lo, hi) / 255.0
    return x + torch.randn_like(x) * std * _mask(B, DEGRADE["noise"]["prob"], dev)

def brightness_contrast(x):
    """Per-sample brightness, then contrast."""
    B = x.shape[0]; dev = x.device
    blo, bhi = DEGRADE["brightness"]["range"]
    mb = _mask(B, DEGRADE["brightness"]["prob"], dev)
    br = torch.empty(B, 1, 1, 1, device=dev).uniform_(blo, bhi)
    x = x * (br * mb + (1 - mb))                       # brightness 1.0 where the switch is off
    clo, chi = DEGRADE["contrast"]["range"]
    mc = _mask(B, DEGRADE["contrast"]["prob"], dev)
    ct = torch.empty(B, 1, 1, 1, device=dev).uniform_(clo, chi) * mc + (1 - mc)
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    return (x - mean) * ct + mean

def colour_cast(x):
    """Per-sample, per-channel gain (cheap-camera white-balance tint)."""
    B = x.shape[0]; dev = x.device
    lo, hi = DEGRADE["color_cast"]["range"]
    m = _mask(B, DEGRADE["color_cast"]["prob"], dev)
    gain = torch.empty(B, 3, 1, 1, device=dev).uniform_(lo, hi)
    return x * (gain * m + (1 - m))

def vignette(x):
    """Per-sample corner darkening."""
    B, _, H, W = x.shape; dev = x.device
    lo, hi = DEGRADE["vignette"]["strength"]
    m = _mask(B, DEGRADE["vignette"]["prob"], dev)
    strength = torch.empty(B, 1, 1, 1, device=dev).uniform_(lo, hi)
    r2 = _radial_r2(H, W, dev)[None, None]
    return x * (1 - strength * m * r2)


def degrade_batch(x):
    """Apply every effect to a [B,3,H,W] batch in [0,1]. Edit the ORDER/SET of effects here if you want."""
    x = low_resolution(x)
    x = blur(x)
    x = camera_tilt(x)
    x = sensor_noise(x)
    x = brightness_contrast(x)
    x = colour_cast(x)
    x = vignette(x)
    return x.clamp(0, 1)


# --- I/O ---
def load_to_tensor(p, size=SIZE):
    img = Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR)
    return torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0

def save_img(t, path):
    arr = (t.clamp(0, 1) * 255).byte().cpu().numpy().transpose(1, 2, 0)
    lo, hi = DEGRADE["jpeg"]["quality"]
    q = random.randint(lo, hi) if random.random() < DEGRADE["jpeg"]["prob"] else 95
    Image.fromarray(arr).save(path, quality=q)

## Preview - fast tuning loop (no files written)

Degrades **10 fixed random images per class** (same images + same random draws every run, via `SEED`),
so when you change `DEGRADE` and re-run this cell, the *only* thing that changes is your edit. Top row of
each pair = clean, bottom = degraded. Nothing is saved to disk - this is just for eyeballing.

In [ ]:
import matplotlib.pyplot as plt

def preview(n_per_class=10, seed=utils.SEED):
    exts = (".jpg", ".jpeg", ".png")
    rng = random.Random(seed)
    picks = {}
    for cls in CLASSES:
        srcs = sorted(p for p in (INPUT_DIR / cls).glob("*") if p.suffix.lower() in exts)
        picks[cls] = rng.sample(srcs, min(n_per_class, len(srcs)))
    flat = [p for cls in CLASSES for p in picks[cls]]

    # Fix BOTH RNGs so the degradation is identical run-to-run; only DEGRADE edits change the result.
    random.seed(seed); torch.manual_seed(seed); np.random.seed(seed)
    deg = degrade_batch(torch.stack([load_to_tensor(p) for p in flat]).to(DEVICE)).cpu()

    ncol = max(len(v) for v in picks.values())
    fig, axes = plt.subplots(2 * len(CLASSES), ncol, figsize=(1.5 * ncol, 1.5 * 2 * len(CLASSES)))
    k = 0
    for r, cls in enumerate(CLASSES):
        for i, p in enumerate(picks[cls]):
            axes[2*r, i].imshow(Image.open(p).convert("RGB").resize((SIZE, SIZE)))
            axes[2*r+1, i].imshow(deg[k].permute(1, 2, 0).numpy()); k += 1
        for ax in axes[2*r]:   ax.set_xticks([]); ax.set_yticks([])
        for ax in axes[2*r+1]: ax.set_xticks([]); ax.set_yticks([])
        axes[2*r, 0].set_ylabel(f"{cls}\nclean", fontsize=9)
        axes[2*r+1, 0].set_ylabel("degraded", fontsize=9)
    plt.tight_layout(); plt.show()

preview()

## Generate the full degraded dataset (auto-balanced)

The biggest class gets `AUGS_FOR_LARGEST` augmentations per image; smaller classes get proportionally
**more** (`round(target / class_size)`), so the classes come out roughly equal in size. One fixed seed
-> reproducible. (The next cell tops them up to *exactly* equal.)

In [ ]:
from tqdm import tqdm

def generate_augmented_dataset():
    reset_degraded_folder()
    torch.manual_seed(utils.SEED); random.seed(utils.SEED); np.random.seed(utils.SEED)
    exts = (".jpg", ".jpeg", ".png")
    sources = {cls: sorted(p for p in (INPUT_DIR / cls).glob("*") if p.suffix.lower() in exts) for cls in CLASSES}
    sizes = {cls: len(sources[cls]) for cls in CLASSES}
    target = max(sizes.values()) * AUGS_FOR_LARGEST
    augs = {cls: max(1, round(target / sizes[cls])) for cls in CLASSES}   # smaller class -> more augs
    print("class sizes :", sizes)
    print("augs / image:", augs, "(smaller classes get more, to balance)")
    for cls in CLASSES:
        out_dir = OUTPUT_DIR / cls
        work = [(p, i) for p in sources[cls] for i in range(augs[cls])]
        print(f"\n{cls}: {sizes[cls]} sources x {augs[cls]} -> {len(work)} degraded")
        for bs in tqdm(range(0, len(work), BATCH)):
            chunk = work[bs:bs + BATCH]
            x = degrade_batch(torch.stack([load_to_tensor(p) for p, _ in chunk]).to(DEVICE))
            for (p, i), t in zip(chunk, x):
                save_img(t, out_dir / f"{p.stem}_aug{i}.jpg")

generate_augmented_dataset()
print("\nDONE.")
for cls in CLASSES:
    print(cls, len(list((OUTPUT_DIR / cls).glob('*'))))

## Balance to exactly equal (top up the smaller classes)

The adaptive augs above get the classes *close*; this makes them identical by adding a few extra degraded copies to whichever classes are short.

In [ ]:
def upsample_to_max():
    random.seed(utils.SEED); torch.manual_seed(utils.SEED)
    exts = (".jpg", ".jpeg", ".png")
    counts = {cls: len(list((OUTPUT_DIR / cls).glob('*'))) for cls in CLASSES}
    target = max(counts.values())
    print("before:", counts, "| target:", target)
    for cls in CLASSES:
        srcs = sorted(p for p in (INPUT_DIR / cls).glob("*") if p.suffix.lower() in exts)
        need = target - counts[cls]
        if need <= 0 or not srcs:
            continue
        picks = [random.choice(srcs) for _ in range(need)]
        for bs in range(0, need, BATCH):
            chunk = picks[bs:bs + BATCH]
            x = degrade_batch(torch.stack([load_to_tensor(p) for p in chunk]).to(DEVICE))
            for j, (p, t) in enumerate(zip(chunk, x)):
                save_img(t, OUTPUT_DIR / cls / f"{p.stem}_extra{bs + j}.jpg")
    print("after :", {cls: len(list((OUTPUT_DIR / cls).glob('*'))) for cls in CLASSES})

upsample_to_max()

## Visual check on the SAVED files

Confirms what actually landed on disk (clean top / a random saved degraded image bottom, per class).

In [ ]:
n = 8
fig, axes = plt.subplots(2 * len(CLASSES), n, figsize=(2 * n, 2 * 2 * len(CLASSES)))
for r, cls in enumerate(CLASSES):
    clean_files = sorted((INPUT_DIR / cls).glob('*'))[:n]
    for i, cp in enumerate(clean_files):
        axes[2*r, i].imshow(Image.open(cp)); axes[2*r, i].axis('off')
        augs = list((OUTPUT_DIR / cls).glob(f"{cp.stem}_aug*.jpg"))
        ax = axes[2*r+1, i]
        ax.imshow(Image.open(random.choice(augs)) if augs else Image.open(cp)); ax.axis('off')
    axes[2*r, 0].set_ylabel(cls, fontsize=12)
plt.tight_layout(); plt.show()